In [2]:
import utils
import numpy as np
from tqdm import tqdm

In [3]:
grid = utils.read_grid("data/grid.pts")
ligand = utils.read_ligand("data/ligand.xyz")

In [4]:
def score_poses(grid_data, ligand, nposes):
    best_energy = 10_000
    best_pose = -1
    for pose in tqdm(range(nposes)):
        posed = utils.transform_ligand(ligand, pose)
        energy = utils.compute_lj_energy(grid_data, posed)
        if energy < best_energy:
          best_energy = energy
          best_pose = pose
    return best_pose, best_energy

In [5]:
best_pose, best_energy = score_poses(grid, ligand, 100000)

100%|██████████| 100000/100000 [00:23<00:00, 4206.19it/s]


In [6]:
print(f"Best pose: {best_pose}")
print(f"Best energy: {best_energy}")

Best pose: 51703
Best energy: -6.387539416409882


CUDA

In [7]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 83.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 13.9 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659448 sha256=2e209d54776ce3b887996444552bf82be2af2eab9de9206ef02e9d223dcbb256
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [ ]:
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import pycuda.gpuarray as gpuarray

In [ ]:
kernel_code = r"""
__device__ double grid_value(
    const double* grid, int n_points_dir,
    int ix, int iy, int iz)
{
    return grid[ix*n_points_dir*n_points_dir + iy*n_points_dir + iz];
}

__device__ double trilinear_interp(
    const double* grid,
    int n_points_dir,
    double x_min, double y_min, double z_min,
    double grid_spacing,
    double x, double y, double z)
{
    double i_f = (x - x_min) / grid_spacing;
    double j_f = (y - y_min) / grid_spacing;
    double k_f = (z - z_min) / grid_spacing;

    int i0 = (int)floor(i_f);
    int j0 = (int)floor(j_f);
    int k0 = (int)floor(k_f);

    i0 = max(0, min(i0, n_points_dir - 2));
    j0 = max(0, min(j0, n_points_dir - 2));
    k0 = max(0, min(k0, n_points_dir - 2));

    int i1 = i0 + 1;
    int j1 = j0 + 1;
    int k1 = k0 + 1;

    double xd = i_f - i0;
    double yd = j_f - j0;
    double zd = k_f - k0;

    double C000 = grid_value(grid,n_points_dir,i0,j0,k0);
    double C100 = grid_value(grid,n_points_dir,i1,j0,k0);
    double C010 = grid_value(grid,n_points_dir,i0,j1,k0);
    double C110 = grid_value(grid,n_points_dir,i1,j1,k0);
    double C001 = grid_value(grid,n_points_dir,i0,j0,k1);
    double C101 = grid_value(grid,n_points_dir,i1,j0,k1);
    double C011 = grid_value(grid,n_points_dir,i0,j1,k1);
    double C111 = grid_value(grid,n_points_dir,i1,j1,k1);

    double C00 = C000 * (1-xd) + C100 * xd;
    double C01 = C001 * (1-xd) + C101 * xd;
    double C10 = C010 * (1-xd) + C110 * xd;
    double C11 = C011 * (1-xd) + C111 * xd;

    double C0 = C00 * (1-yd) + C10 * yd;
    double C1 = C01 * (1-yd) + C11 * yd;

    return C0 * (1-zd) + C1 * zd;
}

extern "C"
__global__ void score_poses(
    const double* grid,
    int n_points_dir,
    double x_min, double y_min, double z_min,
    double grid_spacing,
    const double* ligand,
    const double* centroid,
    int natoms,
    int nposes,
    double* energies)
{
    // TODO: Determine the pose

    double alpha = pose * 0.37;
    double beta  = pose * 0.51;
    double gamma = pose * 0.29;

    double ca = cos(alpha), sa = sin(alpha);
    double cb = cos(beta),  sb = sin(beta);
    double cg = cos(gamma), sg = sin(gamma);

    // Make rotation matrix
    double R00 =  cb*cg;
    double R01 = -cb*sg;
    double R02 =  sb;
    double R10 =  sa*sb*cg + ca*sg;
    double R11 = -sa*sb*sg + ca*cg;
    double R12 = -sa*cb;
    double R20 = -ca*sb*cg + sa*sg;
    double R21 =  ca*sb*sg + sa*cg;
    double R22 =  ca*cb;

    // Get translation in x, y, z, directions
    double tx = 5.0 * sin(pose * 0.21);
    double ty = 5.0 * cos(pose * 0.13);
    double tz = 5.0 * sin(pose * 0.17 + 0.5);

    // TODO: initialize energy

    for (int a = 0; a < natoms; ++a) {
        
        // TODO: transform atom

        // TODO: interpolate energy
    }

    // TODO: store energy
}
"""

In [ ]:
mod = SourceModule(kernel_code)
score_kernel = mod.get_function("score_poses")

In [ ]:
def score_poses_gpu(grid_data, ligand, nposes):
    grid = grid_data["grid"].ravel().astype(np.float64)
    ligand_flat = ligand.astype(np.float64).ravel()
    centroid = ligand.mean(axis=0).astype(np.float64)

    energies = np.empty(nposes, dtype=np.float64)

    # TODO: Create memory on the GPU and copy the data

    block = 128

    # TODO: Determine the grid_dim
    grid_dim = 

    start = cuda.Event()
    end = cuda.Event()

    start.record()
    score_kernel(
        grid_gpu,
        np.int32(grid_data["n_points_dir"]),
        np.float64(grid_data["x_min"]),
        np.float64(grid_data["y_min"]),
        np.float64(grid_data["z_min"]),
        np.float64(grid_data["grid_spacing"]),
        ligand_gpu,
        centroid_gpu,
        np.int32(ligand.shape[0]),
        np.int32(nposes),
        energies_gpu,
        block=(block,1,1),
        grid=(grid_dim,1,1)
    )
    end.record()
    end.synchronize()

    # TODO: Copy data back from GPU to CPU

    return energies, start.time_till(end) * 1e-3

In [16]:
nposes = 100_000

gpu_energies, gpu_time = score_poses_gpu(grid, ligand, nposes)

print("GPU time:", gpu_time, "s")
print("Lowest energy:", np.min(gpu_energies))
print("Best pose id:", int(np.argmin(gpu_energies)))

GPU time: 0.0050769920349121095 s
Lowest energy: -6.387539416409882
Best pose id: 51703
